# Fire Label Validation and Debugging

## Objective

In this notebook, I validate the process of creating wildfire labels by checking how MODIS fire data aligns with the GridMET climate grid.

The goal is to make sure that spatial and temporal matching is working correctly before using these labels in modeling.


In [1]:

import pandas as pd
import numpy as np
import xarray as xr
import os


## Step 1: Load GridMET Grid

Here I load the GridMET dataset for a single year (2018) to inspect:

- Number of latitude and longitude points
- Number of days
- Whether coordinates are ordered correctly

This is important because the fire data will later be mapped onto this grid.

In [2]:

grid_path = "tmmx_2018_CO.nc"  
grid = xr.open_dataset(grid_path)

lats = grid["lat"].values
lons = grid["lon"].values
days = pd.to_datetime(grid["day"].values).normalize()

print("Lat count:", len(lats), "Lon count:", len(lons), "Days:", len(days))
print("Lat ascending?", np.all(np.diff(lats) > 0))
print("Lon ascending?", np.all(np.diff(lons) > 0))


Lat count: 96 Lon count: 169 Days: 365
Lat ascending? False
Lon ascending? True


## Checking Grid Structure

I print out:

- Grid size (lat × lon × time)
- Whether latitude is ascending or descending
- Whether longitude is ascending

GridMET typically has:

- Longitude increasing ✅
- Latitude decreasing ✅

This affects how indexing needs to be handled later.
``

## Step 2: Load MODIS Fire Data

In this step, I load the MODIS fire detection dataset.

This dataset contains:
- Latitude and longitude of fire events
- Acquisition date and time
- Confidence values

I inspect the structure to confirm columns and data types.


In [4]:

fire_csv = "fire_archive_M-C61_731469.csv"
fires = pd.read_csv(fire_csv)

print("Fire rows:", len(fires))
print("Columns:", fires.columns.tolist())
fires.head()


Fire rows: 12211
Columns: ['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'type']


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,type
0,38.8211,-106.5140,311.5,1.6,1.2,2018-01-06,1825,Terra,MODIS,0,6.03,265.9,32.4,D,0
1,37.4673,-105.8486,303.4,1.1,1.0,2018-01-12,1749,Terra,MODIS,56,6.03,278.6,8.3,D,2
2,37.6919,-103.4498,322.8,1.0,1.0,2018-01-12,1749,Terra,MODIS,82,6.03,278.7,24.5,D,0
3,37.6903,-103.4387,305.1,1.0,1.0,2018-01-12,1749,Terra,MODIS,61,6.03,276.8,10.5,D,0
4,37.6896,-103.4591,317.0,2.7,1.6,2018-01-16,2041,Aqua,MODIS,77,6.03,272.2,81.5,D,0


## Step 3: Clean Fire Data

Here I:

- Identify the correct date column
- Convert it to datetime format
- Drop rows with missing date, latitude, or longitude

This ensures that only valid fire records are used.

In [5]:

date_col_candidates = ["acq_date", "acqdate", "acquisition_date", "date"]
date_col = next((c for c in date_col_candidates if c in fires.columns), None)
if date_col is None:
    raise ValueError("Could not find an acquisition date column. Check your CSV columns.")

fires["date"] = pd.to_datetime(fires[date_col], errors="coerce").dt.normalize()


lat_col_candidates = ["latitude", "lat"]
lon_col_candidates = ["longitude", "lon", "long"]

lat_col = next((c for c in lat_col_candidates if c in fires.columns), None)
lon_col = next((c for c in lon_col_candidates if c in fires.columns), None)

if lat_col is None or lon_col is None:
    raise ValueError("Could not find latitude/longitude columns. Check your CSV columns.")

fires = fires.dropna(subset=["date", lat_col, lon_col]).copy()
print("After dropping missing date/lat/lon:", len(fires))

After dropping missing date/lat/lon: 12211



## Step 4: Filter to Colorado

I restrict the fire data to Colorado using a bounding box:

- Longitude: -109.06 to -102.04  
- Latitude: 37.00 to 41.00  

This ensures consistency with the GridMET data used for the model.


In [6]:

lon_min, lon_max = -109.06, -102.04
lat_min, lat_max =  37.00,  41.00

fires = fires[
    (fires[lon_col] >= lon_min) & (fires[lon_col] <= lon_max) &
    (fires[lat_col] >= lat_min) & (fires[lat_col] <= lat_max)
].copy()

print("Fires in Colorado bbox:", len(fires))


Fires in Colorado bbox: 12211


## Step 5: Apply Confidence Filter

MODIS includes a confidence score for each detection.

I keep:
- High-confidence detections (≥ 30)
- Missing values (to avoid excessive data loss)

This helps reduce noise and false positives.


In [7]:

if "confidence" in fires.columns:
    fires["confidence_num"] = pd.to_numeric(fires["confidence"], errors="coerce")
    fires = fires[(fires["confidence_num"].isna()) | (fires["confidence_num"] >= 30)].copy()
    print("After confidence filter (>=30 or missing):", len(fires))


After confidence filter (>=30 or missing): 11453


## Step 6: Map Fire Points to Grid Cells

To align fire events with the grid:

- I use a nearest-neighbor search (`nearest_index_sorted`)
- This converts lat/lon into grid indices

Important:
- Longitude must be ascending
- Latitude may be descending, so I adjust for that

Result:
- Each fire gets assigned a grid cell



## Step 7: Inspect Grid Assignments

I print a few rows showing:

- Original latitude/longitude
- Assigned grid indices (lat_i, lon_i)

This step verifies that spatial mapping is working correctly.


In [9]:
def nearest_index_sorted(arr, x):
    """
    arr must be ascending.
    Returns nearest index for x in arr.
    """
    idx = np.searchsorted(arr, x)
    idx = np.clip(idx, 1, len(arr)-1)
    left = arr[idx-1]
    right = arr[idx]
    return np.where(np.abs(x-left) <= np.abs(x-right), idx-1, idx)

if not np.all(np.diff(lons) > 0):
    raise ValueError("Longitude array is not ascending; unexpected for gridMET.")


lat_descending = np.all(np.diff(lats) < 0)
if lat_descending:
    lats_asc = lats[::-1]  


fire_lons = fires[lon_col].values
fire_lats = fires[lat_col].values

lon_idx = nearest_index_sorted(lons, fire_lons)

if lat_descending:
    lat_idx_from_asc = nearest_index_sorted(lats_asc, fire_lats)
    lat_idx = (len(lats) - 1) - lat_idx_from_asc  
else:
    
    lat_idx = nearest_index_sorted(lats, fire_lats)

fires["lon_i"] = lon_idx.astype(int)
fires["lat_i"] = lat_idx.astype(int)

fires[["date", lat_col, lon_col, "lat_i", "lon_i"]].head()


,date,latitude,longitude,lat_i,lon_i
1,2018-01-12,37.4673,-105.8486,84,77
2,2018-01-12,37.6919,-103.4498,79,135
3,2018-01-12,37.6903,-103.4387,79,135
4,2018-01-16,37.6896,-103.4591,79,134
5,2018-01-16,37.6942,-103.4671,79,134


## Step 8: Match Fire Events with Grid Time Range

I filter fire events to only include dates that exist in the GridMET dataset.

This ensures that:
- Every fire event aligns with a valid grid day
- No out-of-range data is included


In [10]:
# Keep only fires within this grid file's day range (e.g., year 2018)
day_set = set(days)
fires_year = fires[fires["date"].isin(day_set)].copy()

print("Fires matching grid day range:", len(fires_year))
print("Date min/max:", fires_year["date"].min(), fires_year["date"].max())

Fires matching grid day range: 3152
Date min/max: 2018-01-12 00:00:00 2018-12-16 00:00:00



## Step 9: Create Fire Label Array

I create a 3D array:

(day × latitude × longitude)

Each entry represents:
- 1 → fire occurred
- 0 → no fire

Duplicate events are removed to avoid double counting.

In [11]:
n_day = len(days)
n_lat = len(lats)
n_lon = len(lons)

labels = np.zeros((n_day, n_lat, n_lon), dtype=np.uint8)

# Map date -> day index
day_index = {d: i for i, d in enumerate(days)}

# Group unique events to avoid repeatedly setting same cell/day
unique_events = fires_year[["date", "lat_i", "lon_i"]].drop_duplicates()

for d, i_lat, i_lon in unique_events.itertuples(index=False):
    labels[day_index[d], int(i_lat), int(i_lon)] = 1

fire_labels = xr.Dataset(
    data_vars={
        "fire_occurred": (("day", "lat", "lon"), labels)
    },
    coords={
        "day": grid["day"].values,
        "lat": grid["lat"].values,
        "lon": grid["lon"].values
    }
)

fire_labels


<xarray.Dataset>
Dimensions:        (day: 365, lat: 96, lon: 169)
Coordinates:
  * day            (day) datetime64[ns] 2018-01-01 2018-01-02 ... 2018-12-31
  * lat            (lat) float64 40.98 40.94 40.9 40.86 ... 37.11 37.07 37.03
  * lon            (lon) float64 -109.1 -109.0 -109.0 ... -102.1 -102.1 -102.1
Data variables:
    fire_occurred  (day, lat, lon) uint8 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0


## Step 10: Convert to xarray Dataset

The label array is converted into an xarray dataset with:

- Coordinated dimensions (day, lat, lon)
- A variable: `fire_occurred`

This format matches the structure of GridMET climate data and is ready for modeling.


## Step 11: Validate Fire Labels

I compute daily fire counts across the grid to verify:

- How often fires occur
- Whether values look reasonable
- Whether there are any anomalies

This helps confirm that the labeling process is working correctly.


In [12]:
daily_counts = fire_labels["fire_occurred"].sum(dim=("lat","lon")).to_pandas()
daily_counts.describe()


count    365.000000
mean       2.928767
std        5.113586
min        0.000000
25%        0.000000
50%        1.000000
75%        4.000000
max       33.000000
dtype: float64

## Conclusion

In this notebook, I validated the process of generating wildfire labels by:

- Aligning MODIS fire data with GridMET grid cells
- Ensuring correct spatial mapping
- Matching fire events to valid dates
- Creating a structured binary dataset

The results show that:
- Fire events are relatively rare
- Labels are properly aligned with the grid

This step is critical because it ensures that the target variable used in modeling is correct and reliable.
